# RetaRats PubMed Pipeline v2 Runner

Use this notebook while tuning molecules, search rules, Airtable output, and summary behavior. Start with a tiny smoke run, inspect outputs, then scale up.

## 1. Setup

In Colab, run the install cell. Locally, you can skip it if your environment already has the requirements installed.

In [ ]:
# If running in Colab or a fresh local notebook, install dependencies.
from pathlib import Path
import subprocess
import sys

requirements = Path("requirements.txt")
if not requirements.exists():
    requirements = Path("../requirements.txt")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])

In [ ]:
from pathlib import Path
import os
import sys

# Works when the notebook is opened from notebooks/. In Colab after git clone,
# run this notebook from the repo, or adjust REPO_ROOT manually.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

## 2. Credentials and Run Settings

The summary layer is rule-based for now. Leave Airtable blank if you only want local SQLite/CSV output.

In [ ]:
import getpass

def set_secret_env(name, required=False):
    current = os.environ.get(name, "")
    if current:
        print(f"{name}: already set")
        return
    value = getpass.getpass(f"{name}{' (required)' if required else ' (optional)'}: ")
    if value:
        os.environ[name] = value
    elif required:
        raise ValueError(f"{name} is required")

set_secret_env("NCBI_EMAIL", required=True)
set_secret_env("NCBI_API_KEY", required=False)
set_secret_env("AIRTABLE_API_KEY", required=False)
set_secret_env("AIRTABLE_BASE_ID", required=False)

os.environ.setdefault("CONFIG_MODE", "inputs")
os.environ.setdefault("SUMMARY_MODE", "rule_based")
os.environ.setdefault("ROLE_RULES", "config/ROLE_RULES.csv")
os.environ.setdefault("OUTPUT_SINKS", "local")
os.environ.setdefault("LOCAL_DB", "data/retarats_pubmed.sqlite")
os.environ.setdefault("STATE_DB", "data/retarats_state.sqlite")

## 3. Inspect Molecules and Rules

This confirms that `inputs/Moleculessearch.xlsx` and `inputs/Summary Sheet.xlsx` are being read and merged.

In [ ]:
from retarats_pipeline.config import load_config

cfg = load_config(mode="inputs")
print("molecules:", len(cfg.molecules), "rules:", len(cfg.rules))
display(cfg.molecules.head(10))
display(cfg.rules.head(10))

## 4. Live Runner Helper

This streams script output into the notebook so you can see each rule/window and every write batch.

In [ ]:
import subprocess

def run_pipeline(args):
    cmd = [sys.executable, "-u", "retarats_v2.py"] + list(args)
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Pipeline failed with exit code {rc}")
    return rc

## 5. Tiny Smoke Run

Run one molecule first. Use `--refresh-seen` if you want to regenerate local test rows after a previous run.

In [ ]:
run_pipeline([
    "--config-mode", "inputs",
    "--mode", "daily",
    "--daily-days", "30",
    "--molecule", "retatrutide",
    "--max-records-per-rule", "5",
    "--summary-mode", os.environ.get("SUMMARY_MODE", "rule_based"),
    "--sinks", os.environ.get("OUTPUT_SINKS", "local"),
    "--refresh-seen",
])

## 6. Inspect Local Output

The local sink stores JSON payloads in SQLite so you can inspect before sending to Airtable.

In [ ]:
import json
import sqlite3
import pandas as pd

db_path = os.environ.get("LOCAL_DB", "data/retarats_pubmed.sqlite")
conn = sqlite3.connect(db_path)

def payload_table(table, limit=None, connection=None):
    active_conn = connection or sqlite3.connect(os.environ.get("LOCAL_DB", "data/retarats_pubmed.sqlite"))
    sql = f"select payload_json from {table}"
    if limit:
        sql += f" limit {int(limit)}"
    rows = [json.loads(x[0]) for x in active_conn.execute(sql)]
    if connection is None:
        active_conn.close()
    return pd.DataFrame(rows)

molecules = payload_table("molecules")
papers = payload_table("papers")
evidence = payload_table("evidence")
profiles = payload_table("molecule_profiles")

print("molecules", len(molecules), "papers", len(papers), "evidence", len(evidence), "profiles", len(profiles))
display(molecules.head())
display(papers.head())
display(evidence.head())
display(profiles.head())

## 6A. Output Viewer

These cells load the local database into DataFrames, join evidence rows to paper titles/URLs, and show the public/review split.

In [ ]:
def load_outputs():
    conn = sqlite3.connect(os.environ.get("LOCAL_DB", "data/retarats_pubmed.sqlite"))
    tables = {name: payload_table(name, connection=conn) for name in ["molecules", "papers", "evidence", "molecule_profiles"]}
    conn.close()
    ev = tables["evidence"].copy()
    pp = tables["papers"].copy()
    if not ev.empty and not pp.empty and "pmid" in ev.columns and "pmid" in pp.columns:
        keep = [c for c in ["pmid", "title", "abstract", "journal", "pub_date_iso", "pubmed_url", "pubtypes"] if c in pp.columns]
        ev = ev.merge(pp[keep], on="pmid", how="left", suffixes=("", "_paper"))
    tables["evidence_view"] = ev
    return tables

tables = load_outputs()
evidence_view = tables["evidence_view"]
profiles = tables["molecule_profiles"]

def count_table(df, column, n=20):
    if df.empty or column not in df.columns:
        return pd.DataFrame(columns=[column, "n"])
    return df[column].fillna("<blank>").value_counts().head(n).rename_axis(column).reset_index(name="n")

display(count_table(evidence_view, "role_review_bucket"))
display(count_table(evidence_view, "role_category"))
display(count_table(evidence_view, "processing_lane"))
display(count_table(evidence_view, "paper_purpose"))
display(count_table(evidence_view, "evidence_strength_label"))

profile_cols = [c for c in [
    "molecule_id", "molecule_name", "total_evidence_links", "public_candidate_count",
    "curator_review_count", "background_only_count", "exclude_noise_count",
    "strongest_evidence_level", "top_role_categories", "top_therapeutic_area_tags",
    "concise_evidence_profile",
] if c in profiles.columns]
if profile_cols:
    display(profiles[profile_cols].sort_values(["public_candidate_count", "curator_review_count"], ascending=False).head(30))

## 6B. Filtered Evidence Views

Change the filter variables, then rerun this cell.

In [ ]:
VIEW_MOLECULE = ""        # e.g. "retatrutide", "glutathione", "metformin"
VIEW_BUCKET = ""          # public_candidate, curator_review, background_only, exclude_noise
VIEW_LANE = ""            # human_intervention, mechanism_or_pathway, biomarker_or_readout, etc.
VIEW_CATEGORY = ""        # direct_intervention, pathway_component, assay_or_detection, etc.
MAX_ROWS = 50

tables = load_outputs()
df = tables["evidence_view"].copy()
if VIEW_MOLECULE and "molecule_id" in df.columns:
    df = df[df["molecule_id"].astype(str).eq(VIEW_MOLECULE)]
if VIEW_BUCKET and "role_review_bucket" in df.columns:
    df = df[df["role_review_bucket"].astype(str).eq(VIEW_BUCKET)]
if VIEW_LANE and "processing_lane" in df.columns:
    df = df[df["processing_lane"].astype(str).eq(VIEW_LANE)]
if VIEW_CATEGORY and "role_category" in df.columns:
    df = df[df["role_category"].astype(str).eq(VIEW_CATEGORY)]

view_cols = [c for c in [
    "molecule_id", "processing_lane", "paper_purpose", "what_it_is", "role_review_bucket", "role_category", "evidence_strength_label",
    "primary_study_type", "model_type", "molecule_relevance", "role_confidence",
    "condition_tags", "endpoint_tags", "intervention_or_exposure", "comparator_or_control",
    "title", "evidence_summary", "efficacy_signal", "safety_signal", "pubmed_url",
] if c in df.columns]
print("matching rows:", len(df))
display(df[view_cols].head(MAX_ROWS))

## 6C. Review Queues

Use this as the human-review starting point before publishing rows to the website.

In [ ]:
tables = load_outputs()
df = tables["evidence_view"].copy()
review_cols = [c for c in [
    "molecule_id", "role_review_bucket", "role_category", "evidence_strength_label",
    "primary_study_type", "model_type", "title", "evidence_summary", "role_evidence_text", "pubmed_url",
] if c in df.columns]

print("Public candidates")
display(df[df.get("role_review_bucket", "") == "public_candidate"][review_cols].head(25))

print("Needs curator review")
display(df[df.get("role_review_bucket", "") == "curator_review"][review_cols].head(25))

print("Likely non-public/noise examples")
display(df[df.get("role_review_bucket", "") == "exclude_noise"][review_cols].head(25))

## 7. Larger Run

When the smoke run looks good, remove the molecule filter or switch sinks to `local,airtable`.

In [ ]:
# Example: all active rules from the last 7 days, local only.
# Change --sinks to local,airtable once your Airtable table fields exist.
run_pipeline([
    "--config-mode", "inputs",
    "--mode", "daily",
    "--daily-days", "7",
    "--summary-mode", os.environ.get("SUMMARY_MODE", "rule_based"),
    "--sinks", os.environ.get("OUTPUT_SINKS", "local"),
])

## 8. Clean 2026 Refresh

Run this after the smoke test when you want to regenerate the current 2026 records with the new relevance, website_include, evidence_summary, role, and molecule profile fields. This may take a bit longer because it touches every active rule for 2026.


In [ ]:
run_pipeline([
    "--config-mode", "inputs",
    "--mode", "backfill",
    "--start-year", "2026",
    "--summary-mode", os.environ.get("SUMMARY_MODE", "rule_based"),
    "--sinks", os.environ.get("OUTPUT_SINKS", "local"),
    "--refresh-seen",
])


## 9. Re-score Role Rules

Run this after editing `config/ROLE_RULES.csv`. It updates local SQLite and shows how many records are public candidates, need review, are background-only, or are likely noise.

In [ ]:
subprocess.check_call([
    sys.executable,
    "scripts/characterize_roles.py",
    "--db", os.environ.get("LOCAL_DB", "data/retarats_pubmed.sqlite"),
    "--role-rules", os.environ.get("ROLE_RULES", "config/ROLE_RULES.csv"),
    "--config-mode", os.environ.get("CONFIG_MODE", "inputs"),
])

conn = sqlite3.connect(os.environ.get("LOCAL_DB", "data/retarats_pubmed.sqlite"))
evidence = pd.DataFrame([json.loads(x[0]) for x in conn.execute("select payload_json from evidence")])
display(evidence["role_review_bucket"].value_counts(dropna=False).rename_axis("role_review_bucket").reset_index(name="n"))
display(evidence["role_category"].value_counts(dropna=False).head(20).rename_axis("role_category").reset_index(name="n"))

## 10. Multi-Stage Postprocessing

This is the main local/Colab-compatible postprocessing chain. It reuses the local SQLite database, enriches every evidence row, assigns each row to a processing lane, writes lane-specific CSVs, and refreshes exports.

In [ ]:
subprocess.check_call([
    sys.executable,
    "scripts/run_postprocessing_pipeline.py",
    "--db", os.environ.get("LOCAL_DB", "data/retarats_pubmed.sqlite"),
    "--config-mode", os.environ.get("CONFIG_MODE", "inputs"),
    "--role-rules", os.environ.get("ROLE_RULES", "config/ROLE_RULES.csv"),
])

routes_path = Path("exports/processing_routes_summary.csv")
if routes_path.exists():
    display(pd.read_csv(routes_path))

## 10A. Lane Output Viewer

These files are the handoff points for deeper, lane-specific refinement scripts.

In [ ]:
lane_dir = Path("exports/lanes")
post_dir = Path("exports/postprocessed")
print("Lane files")
for path in sorted(lane_dir.glob("*.csv")):
    df = pd.read_csv(path)
    print(f"{path.name}: {len(df)} rows")

print("\nPostprocessed files")
for path in sorted(post_dir.glob("*.csv")):
    df = pd.read_csv(path)
    print(f"{path.name}: {len(df)} rows")

PREVIEW_FILE = "interventions_refined.csv"  # try mechanisms_refined.csv, biomarkers_refined.csv, reviews_refined.csv
preview_path = post_dir / PREVIEW_FILE
if preview_path.exists():
    display(pd.read_csv(preview_path).head(25))

## 10B. Review Slice Viewer

Review slices are saved PICO/PECO-like filters over the broad database. They behave like filtered decks built from tags: one paper can appear in multiple slices.

In [ ]:
slice_manifest = Path("exports/prisma/review_slice_manifest.csv")
slice_flow = Path("exports/prisma/flow_counts_by_slice.csv")
slice_dir = Path("exports/review_slices")

if slice_manifest.exists():
    manifest = pd.read_csv(slice_manifest)
    display(manifest[["slice_id", "slice_name", "question_type", "records_included", "output_csv"]])

if slice_flow.exists():
    flow = pd.read_csv(slice_flow)
    display(flow.head(40))

SLICE_ID = "retatrutide_obesity_t2d_human_intervention"
slice_path = slice_dir / f"{SLICE_ID}.csv"
if slice_path.exists():
    display(pd.read_csv(slice_path).head(25))

## 11. Export CSVs

In [ ]:
subprocess.check_call([sys.executable, "scripts/export_sqlite_to_csv.py"])
print("Exports written under exports/")
for path in sorted(Path("exports").glob("*.csv")):
    size_mb = path.stat().st_size / 1_000_000
    print(f"{path} ({size_mb:.2f} MB)")

## 12. Open Exported CSVs

Preview the exported files from the notebook. In Colab, uncomment the download lines when you want local copies.

In [ ]:
export_dir = Path("exports")
for name in ["prisma/review_slice_manifest.csv", "prisma/flow_counts_by_slice.csv", "processing_routes_summary.csv", "molecule_profiles.csv", "evidence_review.csv", "evidence_characterized.csv", "evidence.csv", "papers.csv", "molecules.csv"]:
    path = export_dir / name
    if path.exists():
        print("\n", name)
        display(pd.read_csv(path).head(20))

# Colab download helpers:
# from google.colab import files
# files.download("exports/evidence_review.csv")
# files.download("exports/evidence_characterized.csv")
# files.download("exports/molecule_profiles.csv")
# files.download("exports/prisma/review_slice_manifest.csv")